*Before running any cells, be sure to run* `uv sync`

# Activity 2: Model Training and Fine-tuning

In today's activity, we'll be training three text classifiers: 
1. a simple logistic regression classifier, 
2. an MLP classifier, and 
3. a (very small) transformer language model classifier. 

All of the training code has been written for you. However, for each section, the hyperparameters are set to more or less their default values. Your job is to find hyperparameters that maximize each model's performance on a held-out test set. 

Our dataset is composed of spam text messages. The classifiers will predict whether or not a text message is spam (1) or not spam ("ham", 0). We'll load our dataset and split it into train and test splits using `sklearn`'s `train_test_split` function.  

In [1]:
import numpy as np
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

spam_ds = load_dataset("ucirvine/sms_spam", split="train")

x_train, x_test, y_train, y_test = train_test_split(spam_ds["sms"], spam_ds["label"], test_size=0.2, random_state=42)


In [2]:
for i in range(10):
    print(f'Text: {x_train[i]}\n\tLabel: {y_train[i]}')

Text: FREE2DAY sexy St George's Day pic of Jordan!Txt PIC to 89080 dont miss out, then every wk a saucy celeb!4 more pics c PocketBabe.co.uk 0870241182716 £3/wk

	Label: 1
Text: Armand says get your ass over to epsilon

	Label: 0
Text: Lol now I'm after that hot air balloon!

	Label: 0
Text: You know, wot people wear. T shirts, jumpers, hat, belt, is all we know. We r at Cribbs

	Label: 0
Text: Good morning, my Love ... I go to sleep now and wish you a great day full of feeling better and opportunity ... You are my last thought babe, I LOVE YOU *kiss*

	Label: 0
Text: UR GOING 2 BAHAMAS! CallFREEFONE 08081560665 and speak to a live operator to claim either Bahamas cruise of£2000 CASH 18+only. To opt out txt X to 07786200117

	Label: 1
Text: Sorry . I will be able to get to you. See you in the morning.

	Label: 0
Text: Sorry, I'll call later

	Label: 0
Text: I AM AT A PARTY WITH ALEX NICHOLS

	Label: 0
Text: Crazy ar he's married. Ü like gd looking guys not me. My frens like say he's ko

In [3]:
# How balanced are our labels?
sum(y_train) / len(y_train)

0.1314196008073559

The dataset is not very balanced: only ~13% of samples are spam, meaning that always guessing "not spam" would get you 87% accuracy.

For this reason, we'll be evaluating using [F1 score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html): the harmonic mean of precision and recall. If we always guess "not spam", what would our F1 be? 

In [4]:
import numpy as np
from sklearn.metrics import f1_score

print(f1_score(y_test, np.zeros_like(y_test)))

0.0


That's a more reasonable result. You should bear in mind dataset imbalance and F1 when tuning the hyperparameters of each of your models below. 

### Activity Guidelines
This activity is a friendly competition between groups to see who can get the best F1 score for each model type. **The winning team for each section will receive 5 points of extra credit on HW02**. (No team can receive more than 5 points EC, so if the same team wins multiple sections, they just get bragging rights, and those 5 points will go to the second place team.)

To present your score for judging, you need to show unmodified cell output to me/a TA, along with your selected hyperparameters. No screenshots--if you want to save a good run, export the notebook to a PDF with a descriptive title. 

Here's some advice for how to go about finding good hyperparameters: 
1. Have a team member train the model with the default hyperparams. 
2. Reference the linked documentation and perforrm research on kwargs that you might not understand fully. 
3. Come up with a plan for which hyperparameters to focus on and how you will adjust their values. Consider how different hyperparameters will interact with each other. 
4. Work in parallel: each teammate should be training a model, recording their results, and sharing with the rest of the team. You should all be working together to find the optimal training configuration for the model.
5. Be prepared to explain why you chose the values you did for the hyperparameters that you changed. 

## Exercise 1: Simple Sequence Classification with Logistic Regression

We're starting with a simple logistic regression classifier. We won't talk too much about logistic regression here, since we've already discussed this at length in earlier lectures. 

However, we do need to discuss how we'll turn the strings in our dataset into something that our LR model can accept. To do this, we'll be using `sklearn`'s `TfidfVectorizer`. This vectorizer turns a string into a vector the size of your *vocabulary*. Each value in a vector corresponds to the inverse document frequency (IDF)-weighted frequency of a term in your vocabulary (TF). This is a bag-of-words representation, so word order doesn't matter. 

Here are some helpful links for hyperparameter tuning:

[`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

[`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


# Modify these kwargs however you like
vectorizer_kwargs = {
    "stop_words": None,
    "max_df": 1.0,
    "min_df": 1,
    "ngram_range": (1, 3), # for subsequences normally in pairs ([The quick], [quick brown], [brown fox])
    "max_features": 800, # cap on vector size
    "use_idf": False # for measuring how important a word is
}

# Leave these three lines alone
vectorizer = TfidfVectorizer(**vectorizer_kwargs)
x_train_vectors = vectorizer.fit_transform(x_train)
x_test_vectors = vectorizer.transform(x_test)


# You can modify these kwargs however you like!
lr_kwargs = {
    "penalty":'deprecated', 
    "C":1.0, 
    "l1_ratio":0.0, 
    "dual":False, 
    "tol":0.0001, 
    "fit_intercept":True, 
    "intercept_scaling":1, 
    "class_weight":None, 
    "solver":'lbfgs', 
    "max_iter":100, 
    "verbose":0, 
    "warm_start":False, 
    "n_jobs":None
}

# Don't change anything below here in this cell
lr_model = LogisticRegression(random_state=42,**lr_kwargs)

lr_model.fit(x_train_vectors, 
y_train)

y_hat_test = lr_model.predict(x_test_vectors)

print(f"F1: {f1_score(y_test, 
y_hat_test):.4f}")

F1: 0.8118


In [ ]:
print(f"Vocab size: {len(vectorizer.vocabulary_)}")
print(f"Num parameters: {lr_model.coef_.flatten().shape[-1]:,}")

## Exercise 2: Training an MLP Classifier 


Next, we'll train an MLP. To simplify things, we're using the `sklearn` implementation, not `torch`. As for the logistic regression classifier, we're going to vectorize our text using the same technique as above. You are free to choose entirely different hyperparameters for your vectorizer, though. 

You will probably want to look at the documentation for [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html). 

In [33]:
from sklearn.neural_network import MLPClassifier

# Consider the vector dimensionality of your input
# and how that will scale the number of parameters in 
# your MLP. you might want to change these kwargs from 
# what you had before. 

# Modify these kwargs however you like
vectorizer_kwargs = {
    "stop_words": None,
    "max_df": 1.0,
    "min_df": 2,
    "ngram_range": (1, 2),
    "max_features": 20000,
}

# Leave these three lines alone
vectorizer = TfidfVectorizer(**vectorizer_kwargs)
x_train_vectors = vectorizer.fit_transform(x_train)
x_test_vectors = vectorizer.transform(x_test)

# Change or add anything else you want here
mlp_kwargs = {
    "hidden_layer_sizes": (128,64), # 1 hidden layer
    "activation": "relu",
    "tol": 0.5,
}

# Leave the rest of this cell alone
mlp = MLPClassifier(random_state=42, **mlp_kwargs)

mlp.fit(x_train_vectors, y_train)

y_hat_test = mlp.predict(x_test_vectors)

print(f"F1: {f1_score(y_test, y_hat_test):.4f}")

F1: 0.9548


In [ ]:
sum(p.flatten().shape[-1] for p in mlp.coefs_)

In [ ]:
print(f"Vocab size: {len(vectorizer.vocabulary_)}")
print(f"Num parameters: {sum(p.flatten().shape[-1] for p in mlp.coefs_):,}")

## Exercise 3: Fine-tuning a Language Model for Classification

Finally, a language model! 

You'll be finetuning `all-MiniLM-l6-v2`, a very small but popular model for a range of tasks. 

The code below can be slightly intimidating if you haven't worked with `transformers` much before. Don't worry too much about the finer points: you can change hyperparameters without understanding 100% of what's going on here. 

For now, refer to these links: 

[`TrainingArguments`](https://huggingface.co/docs/transformers/v5.0.0rc2/en/main_classes/trainer#transformers.TrainingArguments)


[`Trainer`](https://huggingface.co/docs/transformers/v5.0.0rc2/en/main_classes/trainer#api-reference%20][%20transformers.Trainer)

**Warning:** training this model may take five minutes or more, depending on your machine and the hyperparameters you select. Use your time wisely!

Leave this cell alone.

In [34]:
import evaluate 


# Convert our train/test splits back into Datasets
train_dataset = Dataset.from_dict({"text": x_train, "label": y_train})
test_dataset = Dataset.from_dict({"text": x_test, "label": y_test})


# Create an evaluator so we can evaluate our model after training
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1) # Convert float preds to 0 or 1
    return f1.compute(predictions=predictions, references=labels)

# Map our labels to strings and strings to labels
id2label = {0: "HAM", 1: "SPAM"}
label2id = {"HAM": 0, "SPAM": 1}


You'll change some of the values in `training_args` and `trainer` in the cell below. 

In [38]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import Trainer, TrainingArguments


model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load pretrained model from the HF hub
torch.manual_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id
)

# Load our tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Convert our train/test splits back into Datasets
train_dataset = Dataset.from_dict({"text": x_train, "label": y_train})
test_dataset = Dataset.from_dict({"text": x_test, "label": y_test})


# Pre-tokenize our data
def preprocess(examples):
    return tokenizer(examples["text"], truncation=True)

train_tokenized = train_dataset.map(preprocess, batched=True)
test_tokenized = test_dataset.map(preprocess, batched=True)


# These are arguments passed to the Trainer.
training_args = TrainingArguments(
    # Don't change these 
    seed=42,
    eval_strategy="epoch",
    use_cpu=True,
    # Change or add anything you want below here
    num_train_epochs=5,
    per_device_train_batch_size=5,
    learning_rate=1e-5,
    weight_decay=0.001
    save_strategy="yes", # Don't change this unless you want to save checkpoints to disk
)

# This is what trains our model
trainer = Trainer(
    # Leave these alone
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    # Add anything you want here
)

trainer.train()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,0.174752,0.036763,0.971787
2,0.047762,0.037833,0.974684
3,0.027806,0.028699,0.984424
4,0.019386,0.025425,0.984326
5,0.016642,0.025294,0.981366


TrainOutput(global_step=4460, training_loss=0.044836372537997805, metrics={'train_runtime': 457.6291, 'train_samples_per_second': 48.718, 'train_steps_per_second': 9.746, 'total_flos': 68405777971440.0, 'train_loss': 0.044836372537997805, 'epoch': 5.0})

In [ ]:
# Check out the model architecture and number of parameters
print(model)
print(f"Num parameters: {sum(p.numel() for p in model.parameters()):,}")

Try writing some tough ham or spam in the cell below to test your new classifier!

In [ ]:
# Test your trained model! 

input_text = "Hello! Did you forget your Google password? Click here to reset: http://goog.le/reset"
input_ids = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    predicted_class_id = torch.argmax(model(**input_ids).logits, dim=-1).item()
    print(f"Input: {input_text}\nPrediction: {model.config.id2label[predicted_class_id]}")